In [1]:
# @title
%%html
<div style="
  background-color: #e6eefc;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: center;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 26px;
    font-weight: 600;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    Performer FAVOR# Attention Head Replacement in TinyLlama 1.1B Model – Computation Speed and Approximation Convergence Analysis
  </h1>
</div>


SyntaxError: unterminated string literal (detected at line 3) (183334417.py, line 3)

In [ ]:
# @title
%%html
<div style="
  background-color: #e6eefc;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: center;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 26px;
    font-weight: 600;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    0. Setup
  </h1>
</div>


## **0.1 Imports and model load from repository**


In [2]:
import gc
import os
import sys
import time

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [3]:
if os.path.isdir("/content"):
    REPO_DIR = "/content/Performer-attention-LLM"
    if not os.path.isdir(REPO_DIR):
        !git clone https://github.com/Antoinechss/Performer-attention-LLM.git {REPO_DIR}
else:
    REPO_DIR = os.getcwd()

print(f"Repo directory: {REPO_DIR}")


Repo directory: /Users/youssefsah/Desktop/2A PONTS/S2/Projet_département/performer-attention-llm


In [ ]:
sys.path.insert(0, os.path.join(REPO_DIR, 'performer'))
sys.path.insert(0, os.path.join(REPO_DIR, 'models'))

from performer_attention import PerformerAttentionCore, _HAS_TRITON
from favor_analysis_utils import (
    MixedPerformerAttention,
    approximate_causal_attention_weights,
    build_synthetic_qk_cases,
    causal_softmax_attention,
    collect_model_qk_cases,
    evaluate_divergence_cases,
    load_patched_model,
    load_reference_model_and_tokenizer,
    patch_model_attention_layers,
    seed_everything,
    set_performer_heads,
)

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
PROMPT = "<|user|> How do I get a good night's sleep? </s><|assistant|>"
MAX_NEW_TOKENS = 30
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FEATURE_MAP = "favor_sharp"
NUM_FEATURES = 256
NUM_PERF_HEADS = 4
SEED = 42

seed_everything(SEED)
print(f"Triton kernel loaded: {_HAS_TRITON}")
print(f"Device: {DEVICE}, dtype: {DTYPE}")
print(f"Feature map: {FEATURE_MAP} (FAVOR# / SADERF)")
print("Note: FAVOR# currently uses the unfused path, so speed numbers below are method-plus-implementation numbers.")


SyntaxError: unterminated string literal (detected at line 20) (1480756410.py, line 20)

## **0.2 Class for mixed performer / softmax attention heads**


In [ ]:
print("MixedPerformerAttention is imported from models/favor_analysis_utils.py")
print("It patches Llama attention while preserving the 0-performer-head exact-softmax path.")


## **0.3 Standard and Hybrid performer models setup**

Tune `NUM_PERF_HEADS` to replace 0 to 32 of the attention heads by performer approximation.


In [ ]:
std_model, tokenizer = load_reference_model_and_tokenizer(
    model_id=MODEL,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
perf_model, _ = load_reference_model_and_tokenizer(
    model_id=MODEL,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
patch_model_attention_layers(
    perf_model,
    num_performer_heads=NUM_PERF_HEADS,
    num_features=NUM_FEATURES,
    feature_map=FEATURE_MAP,
)

num_heads = std_model.config.num_attention_heads
prompt_ids = tokenizer(PROMPT, return_tensors="pt")["input_ids"].to(DEVICE)
print(f"Models loaded. Heads replaced: {NUM_PERF_HEADS}/{num_heads} performer heads.")


## **0.4 Correctness Checkpoint Test : Setting 0/32 Performer Heads and observing equivalence**


In [ ]:
set_performer_heads(perf_model, 0)

with torch.no_grad():
    std_out = std_model(input_ids=prompt_ids, use_cache=False)
    perf_out = perf_model(input_ids=prompt_ids, use_cache=False)

std_logits = std_out.logits[0].float()
perf_logits = perf_out.logits[0].float()

max_diff = (std_logits - perf_logits).abs().max().item()
mean_diff = (std_logits - perf_logits).abs().mean().item()
cos_sim = F.cosine_similarity(std_logits, perf_logits, dim=-1).mean().item()

std_probs = F.softmax(std_logits[-1], dim=-1)
perf_probs = F.softmax(perf_logits[-1], dim=-1)
kl = F.kl_div(perf_probs.log(), std_probs, reduction='sum').item()

print("Baseline equivalence test 0 performer heads")
print("")
print(f"Max abs difference on logits   |  {max_diff:.6f}")
print(f"Mean abs difference on logits  |  {mean_diff:.6f}")
print(f"Average Cosine similarity      |  {cos_sim:.6f}")
print(f"KL divergence of last token    |  {kl:.6f}")

set_performer_heads(perf_model, NUM_PERF_HEADS)


In [ ]:
# @title
%%html
<div style="
  background-color: lightgreen;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: left;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 15px;
    font-weight: 400;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    Stats above confirm equivalence of the classic model and the hybrid FAVOR# model when MixedAttentionHead is set to 0 performer heads.
  </h1>
</div>


In [ ]:
# @title
%%html
<div style="
  background-color: #e6eefc;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: center;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 26px;
    font-weight: 600;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    1. Generation Quality Analysis:
    Studying how both models respond to a given prompt
  </h1>
</div>


## **1.1 Generation degradation across layers**

Setting first `n=0,1,2...` layers to full performer attention architecture and rest to softmax attention. Approximation error accumulates across layers.


In [ ]:
print("Layerwise replacement: first n layers set to 32/32 performer heads, rest softmax")
print(f"Each replaced layer uses all {num_heads} heads as performer ({FEATURE_MAP}, M={NUM_FEATURES})
")

with torch.no_grad():
    std_ref = std_model(input_ids=prompt_ids, use_cache=False)
std_logits_ref = std_ref.logits[0, -1].float()
std_probs_ref = F.softmax(std_logits_ref, dim=-1)
std_top1_id = std_logits_ref.argmax().item()

print(f"{'Layers':>8}  {'KL':>8}  {'cos_sim':>8}  {'p(top1)':>8}")
print("-" * 40)

for n_perf_layers in [0, 1, 2, 3, 4, 5, 10, 20, 30]:
    for i, layer in enumerate(perf_model.model.layers):
        if i < n_perf_layers:
            layer.self_attn.set_num_performer_heads(num_heads)
        else:
            layer.self_attn.set_num_performer_heads(0)

    with torch.no_grad():
        out = perf_model(input_ids=prompt_ids, use_cache=False)
    logits = out.logits[0, -1].float()
    probs = F.softmax(logits, dim=-1)
    kl = F.kl_div(probs.log(), std_probs_ref, reduction='sum').item()
    cos = F.cosine_similarity(logits.unsqueeze(0), std_logits_ref.unsqueeze(0)).item()
    p_top1 = probs[std_top1_id].item()

    print(f"{n_perf_layers:>8}  {kl:>8.3f}  {cos:>8.4f}  {p_top1:>8.1%}")

set_performer_heads(perf_model, NUM_PERF_HEADS)


In [ ]:
# @title
%%html
<div style="
  background-color: lightyellow;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: left;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 15px;
    font-weight: 400;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    With a 100% (32/32 heads) FAVOR# architecture, generation quality degrades quickly as approximation error propagates through the model layers.
  </h1>
</div>


## **1.2 Response to prompt: per-token distribution comparison (4/32 heads replaced)**


In [ ]:
set_performer_heads(perf_model, NUM_PERF_HEADS)

print(f"Generation benchmark [{NUM_PERF_HEADS}/{num_heads} performer heads]")
print(" ")
print("Response to prompt : How do I get a good night's sleep?")
print(" ")

W = 14
hdr = f"{'Step':>4}  {'Classic':.<{W}}  {'FAVOR#':.<{W}}  {'p(cls)':>7}  {'p_f#(cls)':>11}  {'KL':>6}"
print(hdr)
print("-" * len(hdr))

current_ids = prompt_ids.clone()
classic_tokens, perf_tokens = [], []
kl_per_step = []

with torch.no_grad():
    for step in range(1, MAX_NEW_TOKENS + 1):
        std_out = std_model(input_ids=current_ids, use_cache=False)
        perf_out = perf_model(input_ids=current_ids, use_cache=False)

        std_logits = std_out.logits[0, -1].float()
        perf_logits = perf_out.logits[0, -1].float()
        std_probs = F.softmax(std_logits, dim=-1)
        perf_probs = F.softmax(perf_logits, dim=-1)

        classic_id = std_logits.argmax().item()
        perf_id = perf_logits.argmax().item()
        classic_p = std_probs[classic_id].item()
        perf_p_cls = perf_probs[classic_id].item()
        kl = F.kl_div(perf_probs.log(), std_probs, reduction='sum').item()

        c_tok = repr(tokenizer.decode([classic_id]))[1:-1]
        p_tok = repr(tokenizer.decode([perf_id]))[1:-1]
        print(f"{step:>4}  {c_tok:<{W}}  {p_tok:<{W}}  {classic_p:>6.1%}  {perf_p_cls:>11.1%}  {kl:>6.2f}")

        classic_tokens.append(classic_id)
        perf_tokens.append(perf_id)
        kl_per_step.append(kl)

        current_ids = torch.cat([current_ids, torch.tensor([[classic_id]], device=DEVICE)], dim=-1)
        if classic_id == tokenizer.eos_token_id:
            break

print("")
print("Classic:  ", tokenizer.decode(classic_tokens, skip_special_tokens=True))
print("FAVOR#:  ", tokenizer.decode(perf_tokens, skip_special_tokens=True))
print(f"Avg KL: {sum(kl_per_step) / max(len(kl_per_step), 1):.3f}")


In [ ]:
# @title
%%html
<div style="
  background-color: lightyellow;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: left;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 15px;
    font-weight: 400;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    With 4/32 heads replaced, FAVOR# stays close to the original softmax model, while replacing more heads increases the mismatch.
  </h1>
</div>


In [ ]:
# @title
%%html
<div style="
  background-color: #e6eefc;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: center;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 26px;
    font-weight: 600;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    2. Computation Speed Analysis
  </h1>
</div>


In [ ]:
_CUDA = torch.cuda.is_available()
_dev = torch.device("cuda" if _CUDA else "cpu")
_TRITON = False

H, D = 32, 64
M_VALS = [128, 256]
REPEATS = 10
scale = D ** -0.25
ELEM_STD = D ** -0.25

performer_cores = {
    m: PerformerAttentionCore(head_dim=D, num_features=m, feature_map=FEATURE_MAP).to(_dev)
    for m in M_VALS
}

def time_fn(fn, repeats=REPEATS):
    if _CUDA:
        torch.cuda.synchronize()
    fn()
    if _CUDA:
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeats):
        fn()
    if _CUDA:
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeats * 1000

print("FAVOR# timing note: decode is benchmarked end-to-end with the current unfused implementation.")


## **2.1 `Prefill` step speed benchmark**


In [ ]:
print(f"Prefill step  |  H={H} D={D}")
print(" ")

SEQ_LENS = [256, 512, 1024, 2048, 4096]
_CW = 12

hdr = f"{'N':>6}  {'Classic (ms)':>12}" + "".join(f"  {'FAVOR# M='+str(m):>{_CW}}" for m in M_VALS) + f"  {'speedup':>8}"
print(hdr)
print("-" * len(hdr))

b1_results = []
with torch.no_grad():
    for N in SEQ_LENS:
        q = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16 if _CUDA else torch.float32)
        k = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16 if _CUDA else torch.float32)
        v = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16 if _CUDA else torch.float32)

        def std_attn():
            w = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * (D ** -0.5), dim=-1)
            return torch.matmul(w, v)

        std_ms = time_fn(std_attn)
        row = f"{N:>6}  {std_ms:>11.2f}"

        e2e_times = {}
        for m in M_VALS:
            core = performer_cores[m]
            fn = lambda c=core: c(q, k, v)
            e2e_times[m] = time_fn(fn)
            row += f"  {e2e_times[m]:>{_CW}.2f}"

        best = min(e2e_times.values())
        speedup = std_ms / best
        row += f"  {speedup:>7.2f}x"
        print(row)
        b1_results.append((N, std_ms, e2e_times, speedup))


## **2.2 `Decode` step speed benchmark**


In [ ]:
print("Decode step")
print("")

CACHE_SIZES = [64, 256, 1024, 4096]
_CW = 12

hdr2 = f"{'Context':>8}  {'Classic':>8}" + "".join(f"  {'FAVOR# M='+str(m):>{_CW}}" for m in M_VALS) + f"  {'speedup':>8}"
print(hdr2)
print("-" * len(hdr2))

with torch.no_grad():
    for N in CACHE_SIZES:
        q_new = torch.randn(1, H, 1, D, device=_dev, dtype=torch.float32)
        k_all = torch.randn(1, H, N, D, device=_dev, dtype=torch.float32)
        v_all = torch.randn(1, H, N, D, device=_dev, dtype=torch.float32)

        def std_decode():
            w = torch.softmax(torch.matmul(q_new, k_all.transpose(-2, -1)) * (D ** -0.5), dim=-1)
            return torch.matmul(w, v_all)

        std_ms = time_fn(std_decode)
        row = f"{N:>8}  {std_ms:>7.3f}"

        best_perf = float('inf')
        for m in M_VALS:
            core = performer_cores[m]
            fn = lambda c=core: c(q_new, k_all, v_all)
            t = time_fn(fn)
            best_perf = min(best_perf, t)
            row += f"  {t:>{_CW}.3f}"

        row += f"  {std_ms / best_perf:>7.2f}x"
        print(row)


## **2.3 Speed comparaison plots**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ns = [r[0] for r in b1_results]
std_times = [r[1] for r in b1_results]
ax.plot(ns, std_times, label='Standard O(N^2)', linewidth=2, color='tab:red')
for m in M_VALS:
    perf_times = [r[2][m] for r in b1_results]
    ax.plot(ns, perf_times, label=f'FAVOR# M={m}', linewidth=2)
ax.set_xlabel('Sequence length N')
ax.set_ylabel('Time (ms)')
ax.set_title('Prefill latency')
ax.legend()
ax.set_yscale('log')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)

ax2 = axes[1]
speedups = [r[3] for r in b1_results]
ax2.bar([str(n) for n in ns], speedups)
ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, label='break-even')
ax2.set_xlabel('Sequence length N')
ax2.set_ylabel('Speedup')
ax2.set_title('FAVOR# speedup factor')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## **2.4 Computation speed equivalence for close M and N dimensions O(NxM) = O(N^2)**


### **2.4.1 Small N regime**


In [ ]:
N_VALS = [32, 64, 128, 256, 512, 1024, 2048, 4096]
E1_M_VALS = [64, 128]

print(f"Fix M, vary N  |  H={H} D={D}")
print("")

hdr = (
    f"{'N':>6}  {'Classic':>8}"
    + "".join(f"  {'F# M='+str(m):>10}" for m in E1_M_VALS)
    + f"  {'Speedup':>8}"
)
print(hdr)
print("-" * len(hdr))

e1_results = []
with torch.no_grad():
    for N in N_VALS:
        q = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16 if _CUDA else torch.float32)
        k = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16 if _CUDA else torch.float32)
        v = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16 if _CUDA else torch.float32)

        def std_attn():
            w = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * (D ** -0.5), dim=-1)
            return torch.matmul(w, v)

        std_ms = time_fn(std_attn)
        row = f"{N:>6}  {std_ms:>7.2f}"

        perf_times = {}
        for m in E1_M_VALS:
            if m not in performer_cores:
                performer_cores[m] = PerformerAttentionCore(head_dim=D, num_features=m, feature_map=FEATURE_MAP).to(_dev)
            core = performer_cores[m]
            fn = lambda c=core: c(q, k, v)
            perf_times[m] = time_fn(fn)
            row += f"  {perf_times[m]:>9.2f}"

        best_perf = min(perf_times.values())
        speedup = std_ms / best_perf
        row += f"  {speedup:>7.2f}x"
        print(row)
        e1_results.append((N, std_ms, perf_times, speedup))


In [ ]:
# @title
%%html
<div style="
  background-color: lightyellow;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: left;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 15px;
    font-weight: 400;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    When N is small, the quadratic cost N^2 is cheap, so FAVOR# offers limited speedup benefit in the unfused implementation.
  </h1>
</div>


### **2.4.2 Large M regime**


In [ ]:
N = 2048
M_VALS = [32, 64, 128, 256, 512, 1024, 2048]

with torch.no_grad():
    q = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16 if _CUDA else torch.float32)
    k = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16 if _CUDA else torch.float32)
    v = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16 if _CUDA else torch.float32)

    def std_attn():
        w = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * (D ** -0.5), dim=-1)
        return torch.matmul(w, v)

    e2_std_ms = time_fn(std_attn)
print(f"Standard attention baseline at N={N}: {e2_std_ms:.2f} ms\n")

print(f"{'M':>6}  {'M/D':>5}  {'M/N':>5}  {'FAVOR#(ms)':>11}  {'Speedup':>8}")
print("-" * 50)

e2_results = []
with torch.no_grad():
    for m in M_VALS:
        if m not in performer_cores:
            performer_cores[m] = PerformerAttentionCore(head_dim=D, num_features=m, feature_map=FEATURE_MAP).to(_dev)
        core = performer_cores[m]
        fn = lambda c=core: c(q, k, v)
        perf_ms = time_fn(fn)
        speedup = e2_std_ms / perf_ms

        print(f"{m:>6}  {m / D:>5.1f}  {m / N:>5.2f}  {perf_ms:>10.2f}  {speedup:>7.2f}x")
        e2_results.append((m, perf_ms, speedup))


In [ ]:
# @title
%%html
<div style="
  background-color: lightyellow;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: left;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 15px;
    font-weight: 400;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    As M increases, FAVOR# cost O(N*M) grows linearly.
    When M reaches N, the cost approaches O(N^2), the same asymptotic regime as standard attention.
  </h1>
</div>


### **2.4.3 Plots for computation speed convergence for close M and N values**


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
e1_ns = [r[0] for r in e1_results]
e1_std = [r[1] for r in e1_results]
ax.plot(e1_ns, e1_std, 'o-', label='Standard O(N^2)', linewidth=2, color='tab:red')
colors = ['tab:blue', 'tab:green', 'tab:purple']
for i, m in enumerate(E1_M_VALS):
    perf = [r[2][m] for r in e1_results]
    ax.plot(e1_ns, perf, 's--', label=f'FAVOR# M={m} ({m // D}xD)', linewidth=2, color=colors[i])
ax.set_xlabel('Sequence length N')
ax.set_ylabel('Time (ms)')
ax.set_title('Latency vs N (log-log)')
ax.legend(fontsize=9)
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for i, m in enumerate(E1_M_VALS):
    speedups = [r[1] / r[2][m] for r in e1_results]
    ax.plot(e1_ns, speedups, 'o-', label=f'M={m} ({m // D}xD)', linewidth=2, color=colors[i])
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, linewidth=1, label='break-even')
ax.set_xlabel('Sequence length N')
ax.set_ylabel('Speedup (std / FAVOR#)')
ax.set_title('Speedup vs N')
ax.legend(fontsize=9)
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

ax = axes[1, 0]
e2_ms = [r[0] for r in e2_results]
e2_perf = [r[1] for r in e2_results]
ax.plot(e2_ms, e2_perf, 's-', label='FAVOR#', linewidth=2, color='tab:blue')
ax.axhline(y=e2_std_ms, color='tab:red', linewidth=2, label=f'Standard (N={N})')
ax.set_xlabel(f'Number of features M (D={D})')
ax.set_ylabel('Time (ms)')
ax.set_title('Latency vs M')
ax.legend(fontsize=9)
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
e2_md_ratios = [m / D for m in e2_ms]
e2_speedups = [r[2] for r in e2_results]
ax.plot(e2_md_ratios, e2_speedups, 'o-', linewidth=2, color='tab:green', markersize=8)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, linewidth=1, label='break-even')
ax.set_xlabel('M / D ratio')
ax.set_ylabel('Speedup (std / FAVOR#)')
ax.set_title('Speedup vs M/D ratio')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

plt.suptitle(f'FAVOR# speed convergence analysis — O(N*M) vs O(N^2), D={D}', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# @title
%%html
<div style="
  background-color: #e6eefc;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: center;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 26px;
    font-weight: 600;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    3. Generation independent attention approximation convergence analysis
  </h1>
</div>


### This section validates the pure FAVOR# mathematical approximation of softmax, independent of the Q and K model vectors. Simulating pretrained model Q/K norms with random vectors scaled by `D^-1/4`.


## **3.0 Defining a standard softmax causal attention to benchmark against**


In [ ]:
print("Loaded exact causal_softmax_attention from models/favor_analysis_utils.py")


## **3.1 Convergence of FAVOR# approximation towards softmax as M/D ratio increases**


In [ ]:
M_VALS = [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
N = 64

torch.manual_seed(SEED)
q = torch.randn(1, 4, N, D, device=_dev, dtype=torch.float32) * ELEM_STD
k = torch.randn(1, 4, N, D, device=_dev, dtype=torch.float32) * ELEM_STD
v = torch.randn(1, 4, N, D, device=_dev, dtype=torch.float32)

with torch.no_grad():
    out_std, _ = causal_softmax_attention(q, k, v)

print(f"\n{'=' * 70}")
print(f"Fix N={N}, D={D}, and increase M/D ratio")
print(f"{'=' * 70}\n")
print(f"{'M':>6}  {'M/D':>5}  {'MSE':>10}  {'Cosine':>8}  {'RelErr%':>8}")
print("-" * 45)

g2_results = []
for m in M_VALS:
    mses, coss, rels = [], [], []
    for trial in range(5):
        torch.manual_seed(SEED + trial)
        core = PerformerAttentionCore(head_dim=D, num_features=m, feature_map=FEATURE_MAP).to(_dev)
        with torch.no_grad():
            out_perf = core(q, k, v)
        mses.append(F.mse_loss(out_perf, out_std).item())
        coss.append(F.cosine_similarity(out_perf.reshape(-1, D), out_std.reshape(-1, D), dim=-1).mean().item())
        rels.append(((out_perf - out_std).norm() / out_std.norm()).item())

    avg_mse = sum(mses) / len(mses)
    avg_cos = sum(coss) / len(coss)
    avg_rel = sum(rels) / len(rels)

    print(f"{m:>6}  {m / D:>5.1f}  {avg_mse:>10.6f}  {avg_cos:>8.4f}  {avg_rel * 100:>7.2f}%")
    g2_results.append((m, avg_mse, avg_cos, avg_rel))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

ax = axes[0]
g2_md = [r[0] / D for r in g2_results]
g2_rels = [r[3] * 100 for r in g2_results]
ax.plot(g2_md, g2_rels, 'o-', linewidth=2, color='tab:blue', label='Measured RelErr%', markersize=7)
ax.set_xlabel(f'M / D ratio (D={D})')
ax.set_ylabel('Relative L2 error %')
ax.set_title('Error decreases as M/D grows')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
g2_coss = [r[2] for r in g2_results]
ax.plot(g2_md, g2_coss, 's-', linewidth=2, color='tab:green', markersize=7)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Perfect match')
ax.axhline(y=0.95, color='orange', linestyle=':', alpha=0.5, label='0.95 threshold')
ax.set_xlabel(f'M / D ratio (D={D})')
ax.set_ylabel('Cosine similarity')
ax.set_title('Cosine similarity converges to 1.0')
ax.set_xscale('log', base=2)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('FAVOR# approximation pre-generation precision', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# @title
%%html
<div style="
  background-color: lightyellow;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: left;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 15px;
    font-weight: 400;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    As more features are sampled, the FAVOR# approximation converges towards softmax. Error decreases and cosine similarity converges to 1.
  </h1>
</div>


## **3.2 Attention weight comparaison – pattern similarity**


In [ ]:
torch.manual_seed(SEED)
N_vis = 32
q = torch.randn(1, 1, N_vis, D, device=_dev, dtype=torch.float32) * ELEM_STD
k = torch.randn(1, 1, N_vis, D, device=_dev, dtype=torch.float32) * ELEM_STD
v = torch.randn(1, 1, N_vis, D, device=_dev, dtype=torch.float32)

with torch.no_grad():
    _, w_std = causal_softmax_attention(q, k, v)
w_std = w_std[0, 0].cpu()

core_vis = PerformerAttentionCore(head_dim=D, num_features=512, feature_map=FEATURE_MAP).to(_dev)
with torch.no_grad():
    w_perf = approximate_causal_attention_weights(core_vis, q, k)[0, 0].cpu()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

im0 = axes[0].imshow(w_std.numpy(), cmap='Blues', aspect='auto')
axes[0].set_title('Standard softmax attention')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(w_perf.numpy(), cmap='Blues', aspect='auto')
axes[1].set_title(f'FAVOR# attention (M=512, M/D={512 // D})')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.suptitle('Comparison of attention patterns')
plt.tight_layout()
plt.show()

frob = torch.norm(w_std - w_perf, p='fro').item()
print(f"Frobenius norm ||W_softmax - W_favor#||_F = {frob:.6f}")


In [ ]:
# @title
%%html
<div style="
  background-color: lightyellow;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: left;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 15px;
    font-weight: 400;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    Attention weight patterns remain similar between FAVOR# and the exact softmax baseline.
  </h1>
</div>


In [ ]:
# @title
%%html
<div style="
  background-color: #e6eefc;
  padding: 20px 24px;
  border-radius: 12px;
  text-align: center;
">
  <h1 style="
    margin: 0;
    color: #000000;
    font-size: 26px;
    font-weight: 600;
    font-family: Arial, sans-serif;
    line-height: 1.4;
  ">
    4. Divergence Study
  </h1>
</div>


## **4.1 Synthetic Q/K stress cases**


In [ ]:
synthetic_cases = build_synthetic_qk_cases(seq_len=64, head_dim=D, seed=SEED)
synthetic_results = evaluate_divergence_cases(
    synthetic_cases,
    feature_map=FEATURE_MAP,
    num_features=NUM_FEATURES,
    seed=SEED,
    num_draws=1024,
    num_trials=128,
    alpha=0.05,
)

print(f"{'Rank':>4}  {'Case':>28}  {'Row':>4}  {'JS':>10}  {'RelErr':>10}  {'p':>9}  {'q':>9}")
print("-" * 82)
for rank, row in enumerate(synthetic_results[:5], start=1):
    print(
        f"{rank:>4}  {row['case'][:28]:>28}  {row['worst_row']:>4}"
        f"  {row['max_row_js']:>10.6f}  {row['max_row_relative_output_error']:>10.6f}"
        f"  {row['p_value']:>9.4f}  {row['q_value']:>9.4f}"
    )


## **4.2 Model-extracted Q/K stress cases**


In [ ]:
capture_model, _ = load_patched_model(
    model_id=MODEL,
    num_performer_heads=0,
    num_features=NUM_FEATURES,
    feature_map='favor_plus',
    torch_dtype=DTYPE,
    device_map=DEVICE,
    capture_qkv=True,
)

layer_indices = [idx for idx in [0, 1, 2, 4, 8, 16, 24, 31] if idx < len(capture_model.model.layers)]
model_cases = collect_model_qk_cases(
    capture_model,
    tokenizer,
    prompts=[PROMPT],
    device=torch.device(DEVICE),
    layer_indices=layer_indices,
    head_indices=[0, 1, 2, 3],
)

model_results = evaluate_divergence_cases(
    model_cases,
    feature_map=FEATURE_MAP,
    num_features=NUM_FEATURES,
    seed=SEED,
    num_draws=1024,
    num_trials=128,
    alpha=0.05,
)

print(f"{'Rank':>4}  {'Layer':>5}  {'Head':>4}  {'Row':>4}  {'JS':>10}  {'RelErr':>10}  {'p':>9}  {'q':>9}")
print("-" * 78)
for rank, row in enumerate(model_results[:5], start=1):
    print(
        f"{rank:>4}  {row['layer']:>5}  {row['head']:>4}  {row['worst_row']:>4}"
        f"  {row['max_row_js']:>10.6f}  {row['max_row_relative_output_error']:>10.6f}"
        f"  {row['p_value']:>9.4f}  {row['q_value']:>9.4f}"
    )

del capture_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
